In [9]:
library(tidyverse)
library(lubridate)

# Load hcris data
final.hcris.v2010 <- read_tsv("../../hwk4/data/input/HCRIS_Data_v2010.txt")

# Clean HCRIS
final.hcris <- final.hcris.v2010 %>%
  # Calculate uncompensated care charges
  mutate(uncomp_care = tot_uncomp_care_charges - tot_uncomp_care_partial_pmts + bad_debt) %>%
  # Standardize dates and enforce absolute values
  mutate(fy_end = mdy(fy_end), 
         fy_start = mdy(fy_start),
         date_processed = mdy(date_processed), 
         date_created = mdy(date_created), 
         tot_discounts = abs(tot_discounts), 
         hrrp_payment = abs(hrrp_payment)) %>%
  mutate(fyear = year(fy_end)) %>%
  arrange(provider_number, fyear) %>%
  select(-year)

# Create the State Crosswalk (to include DC)
st_crosswalk <- tibble(
  state_name = c(state.name, "District of Columbia"),
  st_abb = c(state.abb, "DC")
)

# Clean KFF data using the Crosswalk
kff.final <- read_csv("../data/input/kff-data.csv", skip = 2) %>%
  rename(state_name = 1, status = 2, expansion_date = 3) %>%
  # Merge with crosswalk to get abbreviations (st_abb)
  left_join(st_crosswalk, by = "state_name") %>%
  mutate(
    expansion_year = year(mdy(expansion_date))
  ) %>%
  filter(!is.na(st_abb))

# Merge HCRIS and KFF
final_analysis_data <- final.hcris %>%
  left_join(kff.final, by = c("state" = "st_abb")) %>%
  mutate(

    uncomp_millions = uncomp_care / 1000000,
    expand = ifelse(!is.na(expansion_year) & fyear >= expansion_year, 1, 0)
  ) %>%
  filter(uncomp_millions > 0, uncomp_millions < 250)

write_rds(final_analysis_data, "../data/output/hcris_kff_combined.rds")

Rows: 85496 Columns: 48
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): provider_number, fy_start, fy_end, date_processed, date_created, d...
dbl (35): report, status, year, beds, tot_charges, tot_discounts, net_pat_re...
lgl  (1): npi

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 52 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): Location, Status of Medicaid Expansion Decision, Expansion Implemen...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `expansion_year = year(mdy(expansion_date))`.
Caused by warning:
!  10 failed to parse.”
